# Yambda 02 · Preprocessing and temporal split

Decisions from `01_eda`:

| Decision | Choice |
|---|---|
| Valid listen (training signal) | played ≥ 50%, organic **and** recommended |
| Main eval target | organic + valid listen + never heard in train |
| Secondary eval target | valid listen + never heard (recommended discoveries included) |
| "Already heard" | any listen in train, skips included |
| Users | ≥ 10 distinct vocabulary tracks in train |
| Cut points | val: train < day 273, eval 273–287 · test: train < day 287, eval 287–301 |
| Tracks (min listeners) | **5**, chosen from the threshold table on the validation split |

Code: `src/musicrec/yambda.py` (event table) and `src/musicrec/preprocess.py` (`make_split`).

In [1]:
import pandas as pd
from musicrec import config
from musicrec.yambda import build_events, DAY
from musicrec.preprocess import item_threshold_table, make_split, extra_targets

pd.set_option("display.float_format", "{:,.3f}".format)
events = build_events(min_played_ratio=50)
events.head()

,user_id,ts,track_key,is_organic,played_ratio,is_full_listen
0,100,39420,8326270,False,100,True
1,100,39420,1441281,False,100,True
2,100,39625,286361,False,100,True
3,100,40110,732449,False,100,True
4,100,40360,3397170,False,46,False


## How many listeners must a track have to enter the vocabulary?

Reading the table: going down the rows (higher threshold) the vocabulary shrinks, the share of
training interactions we keep drops slowly (popular tracks carry most of the listening), and the
share of future discoveries the model is *able* to recommend drops too. Tracks below the
threshold are not deleted from anyone's history: they still count as "already heard".

In [2]:
table = item_threshold_table(
    events, config.YAMBDA_VAL_TRAIN_END, config.YAMBDA_VAL_EVAL_END,
    thresholds=(1, 2, 3, 5, 10, 20, 50),
    positive="is_full_listen", target="is_organic",
)
table

154,703 (user, new track) pairs in the eval window; 13.3% are tracks nobody interacted with during training (pure cold start)


,min_listeners,n_items,train interactions kept,future discoveries recommendable
0,1,585868,1.000,0.867
1,2,307507,0.973,0.830
2,3,227848,0.957,0.804
3,5,160613,0.935,0.771
4,10,101925,0.895,0.713
5,20,63027,0.837,0.646
6,50,30085,0.719,0.527


In [ ]:
MIN_ITEM_USERS = 5    # chosen from the table above: 77% of discoveries recommendable (vs 71% at 10)
MIN_USER_ITEMS = 10

splits = {}
for name, (t0, t1), out in [
    ("val",  (config.YAMBDA_VAL_TRAIN_END,  config.YAMBDA_VAL_EVAL_END),  config.YAMBDA_SPLIT_VAL_DIR),
    ("test", (config.YAMBDA_TEST_TRAIN_END, config.YAMBDA_TEST_EVAL_END), config.YAMBDA_SPLIT_TEST_DIR),
]:
    s = make_split(events, t0, t1, MIN_ITEM_USERS, MIN_USER_ITEMS,
                   positive="is_full_listen", target="is_organic")
    s.save(out)
    # secondary target: valid new listens, organic OR recommended
    extra_targets(events, s, positive="is_full_listen").to_parquet(out / "targets_all.parquet", index=False)
    splits[name] = s

pd.DataFrame({k: s.stats for k, s in splits.items()})

## Sanity checks

In [4]:
for name, s in splits.items():
    overlap = s.targets.merge(s.train_counts, on=["user_idx", "item_idx"])
    assert overlap.empty, f"{name}: {len(overlap)} targets leak from train"
    assert s.train_events["ts"].max() < s.train_end
    assert s.train_counts["user_idx"].between(0, s.n_users - 1).all()
    assert s.train_counts["item_idx"].between(0, s.n_items - 1).all()
    assert s.targets["item_idx"].between(0, s.n_items - 1).all()
    print(f"{name}: all checks passed")

val: all checks passed
test: all checks passed


In [5]:
s = splits["val"]
print("organic targets per evaluated user:")
s.targets.groupby("user_idx").size().describe(percentiles=[0.1, 0.5, 0.9])

organic targets per evaluated user:


count   5,580.000
mean       17.680
std        27.629
min         1.000
10%         1.000
50%         9.000
90%        43.000
max       665.000
dtype: float64